# 🔗 Clase 1: Introducción a LangChain

## Bienvenido a la Semana 2, Clase 1

En esta clase aprenderás:
- ✅ ¿Por qué necesitamos un orquestador?
- ✅ LangChain: Framework para aplicaciones con LLMs
- ✅ Conceptos: Chains, Runnables, LCEL
- ✅ Paralelización vs ejecución secuencial
- ✅ Integración con FastAPI
- ✅ LangGraph: Introducción a grafos

---

In [ ]:
# Instalación
!pip install langchain langchain-openai langsmith python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough, RunnableParallel

load_dotenv()

# Configurar LangSmith (opcional)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "taller-ia-semana2"

llm = ChatOpenAI(model="gpt-4", temperature=0.7)
print("✅ LangChain configurado")

## 🤔 Parte 1: ¿Por qué LangChain?

### Problemas sin Orquestador

```python
# ❌ Código repetitivo
response1 = openai.chat.completions.create(...)
response2 = openai.chat.completions.create(...)
response3 = openai.chat.completions.create(...)

# ❌ Difícil de mantener
# ❌ Sin reutilización
# ❌ Difícil de testear
```

### Con LangChain

```python
# ✅ Componentes reutilizables
# ✅ Fácil de componer
# ✅ Debugging integrado
# ✅ Ecosistema completo
```

## 🔗 Parte 2: Chains Básicas

Una **chain** es una secuencia de operaciones.

In [ ]:
# Chain simple: Prompt → LLM → Output Parser
prompt = ChatPromptTemplate.from_template("Cuenta un chiste sobre {tema}")
output_parser = StrOutputParser()

# Crear chain usando LCEL (LangChain Expression Language)
chain = prompt | llm | output_parser

# Ejecutar
resultado = chain.invoke({"tema": "programadores"})
print(resultado)

### LCEL: LangChain Expression Language

El operador `|` (pipe) conecta componentes:

```python
chain = component1 | component2 | component3
```

Es como un pipeline de Unix!

In [ ]:
# Chain con múltiples pasos
from langchain.prompts import ChatPromptTemplate

# Paso 1: Generar idea
idea_prompt = ChatPromptTemplate.from_template(
    "Dame una idea innovadora para una startup de {industria}"
)

# Paso 2: Analizar la idea
analisis_prompt = ChatPromptTemplate.from_template(
    "Analiza esta idea de startup y dame 3 pros y 3 contras:\n\n{idea}"
)

# Chain completa
chain_idea = idea_prompt | llm | StrOutputParser()
chain_analisis = analisis_prompt | llm | StrOutputParser()

# Ejecutar secuencialmente
idea = chain_idea.invoke({"industria": "educación"})
print("💡 Idea generada:")
print(idea)
print("\n" + "="*80 + "\n")

analisis = chain_analisis.invoke({"idea": idea})
print("📊 Análisis:")
print(analisis)

## 🔄 Parte 3: Paralelización vs Secuencial

### Ejecución Secuencial

```
A → B → C → Resultado
```

### Ejecución Paralela

```
    ┌─ A ─┐
    ├─ B ─┤ → Combinar → Resultado
    └─ C ─┘
```

In [ ]:
# Ejemplo: Analizar un producto desde múltiples perspectivas
producto = "iPhone 15 Pro"

# Crear prompts para diferentes análisis
prompt_tecnico = ChatPromptTemplate.from_template(
    "Analiza las especificaciones técnicas de {producto}"
)
prompt_precio = ChatPromptTemplate.from_template(
    "Analiza la relación calidad-precio de {producto}"
)
prompt_competencia = ChatPromptTemplate.from_template(
    "Compara {producto} con su competencia"
)

# Ejecutar en paralelo usando RunnableParallel
analisis_paralelo = RunnableParallel(
    tecnico=prompt_tecnico | llm | StrOutputParser(),
    precio=prompt_precio | llm | StrOutputParser(),
    competencia=prompt_competencia | llm | StrOutputParser()
)

import time
inicio = time.time()
resultados = analisis_paralelo.invoke({"producto": producto})
tiempo = time.time() - inicio

print(f"⏱️ Tiempo de ejecución: {tiempo:.2f}s\n")
print("📊 Análisis Técnico:")
print(resultados["tecnico"][:200] + "...\n")
print("💰 Análisis de Precio:")
print(resultados["precio"][:200] + "...\n")
print("🏆 Análisis de Competencia:")
print(resultados["competencia"][:200] + "...")

## 🎯 Parte 4: Runnables Avanzados

### RunnablePassthrough

Pasa datos sin modificar:

In [ ]:
# Ejemplo: Mantener el input original
from langchain.schema.runnable import RunnablePassthrough

prompt = ChatPromptTemplate.from_template(
    "Traduce al inglés: {texto}"
)

chain_con_original = {
    "original": RunnablePassthrough(),
    "traduccion": prompt | llm | StrOutputParser()
}

resultado = chain_con_original["traduccion"].invoke({"texto": "Hola mundo"})
print(f"Original: Hola mundo")
print(f"Traducción: {resultado}")

### RunnableLambda

Ejecuta funciones personalizadas:

In [ ]:
from langchain.schema.runnable import RunnableLambda

def contar_palabras(texto: str) -> dict:
    """Cuenta palabras en el texto."""
    return {
        "texto": texto,
        "palabras": len(texto.split()),
        "caracteres": len(texto)
    }

# Chain con función personalizada
chain = (
    prompt 
    | llm 
    | StrOutputParser() 
    | RunnableLambda(contar_palabras)
)

resultado = chain.invoke({"tema": "inteligencia artificial"})
print(f"Palabras: {resultado['palabras']}")
print(f"Caracteres: {resultado['caracteres']}")

## 🌐 Parte 5: Integración con FastAPI

Ver el archivo `fastapi_app/main.py` para una implementación completa.

Ejemplo básico:

In [ ]:
# Código de ejemplo (no ejecutar en notebook)
ejemplo_fastapi = '''
from fastapi import FastAPI
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser

app = FastAPI()
llm = ChatOpenAI()

@app.post("/chat")
async def chat(message: str):
    prompt = ChatPromptTemplate.from_template("{input}")
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({"input": message})
    return {"response": response}
'''

print("📝 Ejemplo de integración con FastAPI:")
print(ejemplo_fastapi)

## 📊 Parte 6: LangGraph - Introducción

**LangGraph** permite crear flujos más complejos con:
- Decisiones condicionales
- Loops
- Estado compartido

```
LangChain: A → B → C (lineal)
LangGraph: A → B → ¿condición? → C o D (con lógica)
```

Lo veremos en profundidad en Semana 3.

## 💡 Ejercicios Prácticos

In [ ]:
# Ejercicio 1: Crea una chain que:
# 1. Genere un título de blog sobre un tema
# 2. Genere 3 subtítulos para ese blog
# 3. Escriba la introducción

# 👉 Tu código aquí
tema = "Machine Learning para principiantes"

# Paso 1: Título
prompt_titulo = ChatPromptTemplate.from_template(
    "Genera un título atractivo para un blog sobre: {tema}"
)

# Paso 2: Subtítulos
prompt_subtitulos = ChatPromptTemplate.from_template(
    "Para el blog titulado '{titulo}', genera 3 subtítulos interesantes"
)

# Paso 3: Introducción
prompt_intro = ChatPromptTemplate.from_template(
    "Escribe una introducción de 2 párrafos para un blog titulado '{titulo}'"
)

# Ejecutar
titulo = (prompt_titulo | llm | StrOutputParser()).invoke({"tema": tema})
print(f"📝 Título: {titulo}\n")

subtitulos = (prompt_subtitulos | llm | StrOutputParser()).invoke({"titulo": titulo})
print(f"📋 Subtítulos:\n{subtitulos}\n")

intro = (prompt_intro | llm | StrOutputParser()).invoke({"titulo": titulo})
print(f"✍️ Introducción:\n{intro}")

In [ ]:
# Ejercicio 2: Análisis paralelo de sentimiento
# Analiza el mismo texto desde 3 perspectivas: positivo, negativo, neutral

texto_analizar = "El nuevo producto es innovador pero muy caro."

# 👉 Crea un RunnableParallel que analice desde las 3 perspectivas
analisis_sentimiento = RunnableParallel(
    positivo=ChatPromptTemplate.from_template(
        "Identifica los aspectos POSITIVOS de: {texto}"
    ) | llm | StrOutputParser(),
    negativo=ChatPromptTemplate.from_template(
        "Identifica los aspectos NEGATIVOS de: {texto}"
    ) | llm | StrOutputParser(),
    neutral=ChatPromptTemplate.from_template(
        "Da un análisis OBJETIVO de: {texto}"
    ) | llm | StrOutputParser()
)

resultados = analisis_sentimiento.invoke({"texto": texto_analizar})
for clave, valor in resultados.items():
    print(f"\n{clave.upper()}:")
    print(valor)

## 🎓 Resumen

### Conceptos Clave

1. **LangChain**: Framework para orquestar LLMs
2. **LCEL**: Sintaxis con `|` para conectar componentes
3. **Chains**: Secuencias de operaciones
4. **Runnables**: Componentes ejecutables
5. **Paralelización**: Ejecutar múltiples operaciones simultáneamente
6. **LangSmith**: Debugging y monitoreo

### Próxima Clase

En **Clase 2** aprenderemos:
- 🤖 Agentes que toman decisiones
- 🛠️ Tools (herramientas)
- 🧠 ReAct reasoning
- 🌐 Búsqueda web con Tavily

---

**¡Nos vemos en la próxima clase! 🚀**